# Training Notebook for trend_intelligence/train_trend.py

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import psycopg2
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import json
import joblib
import os
from datetime import datetime, timedelta

# Connection details
DB_DB = "postgres"
DB_USER = "postgres"
DB_PASS = "Madhavan@2004@"
DB_HOST = "db.lpoaswfbufnosziwhtsq.supabase.co"
DB_PORT = "5432"

def main():
    print("--- STARTING TREND INTELLIGENCE K-FOLD TRAINING (LightGBM) ---")
    
    # 1. Load Data
    conn = psycopg2.connect(host=DB_HOST, database=DB_DB, user=DB_USER, password=DB_PASS, port=DB_PORT)
    sales_df = pd.read_sql("SELECT transaction_date, store_id, product_id, units_sold, promotion_flag FROM public.sales_transactions", conn)
    products_df = pd.read_sql("SELECT product_id, category_id, product_name FROM public.products", conn)
    trends_df = pd.read_sql("SELECT category_id, trend_score, momentum FROM public.trends", conn)
    conn.close()

    # 2. Daily Aggregation
    sales_df['transaction_date'] = pd.to_datetime(sales_df['transaction_date'])
    daily_sales = sales_df.groupby(['store_id', 'product_id', 'transaction_date'])['units_sold'].sum().reset_index()
    daily_sales = daily_sales.merge(products_df[['product_id', 'category_id']], on='product_id')
    
    # 3. Vectorized Time-Series Engineering
    daily_sales = daily_sales.sort_values(['store_id', 'product_id', 'transaction_date'])
    grouped = daily_sales.groupby(['store_id', 'product_id'])['units_sold']
    
    daily_sales['roll_7d'] = grouped.transform(lambda x: x.rolling(window=7, min_periods=7).mean())
    daily_sales['roll_30d'] = grouped.transform(lambda x: x.rolling(window=30, min_periods=30).mean())
    
    daily_sales['prev_roll_7d'] = grouped.transform(lambda x: x.shift(7).rolling(window=7, min_periods=7).mean())
    daily_sales['rolling_sales_growth_7d'] = (daily_sales['roll_7d'] - daily_sales['prev_roll_7d']) / (daily_sales['prev_roll_7d'] + 0.1)
    
    daily_sales['prev_roll_30d'] = grouped.transform(lambda x: x.shift(30).rolling(window=30, min_periods=30).mean())
    daily_sales['rolling_sales_growth_30d'] = (daily_sales['roll_30d'] - daily_sales['prev_roll_30d']) / (daily_sales['prev_roll_30d'] + 0.1)
    
    daily_sales['sales_acceleration_rate'] = daily_sales.groupby(['store_id', 'product_id'])['rolling_sales_growth_7d'].diff(periods=7)
    
    # 4. Contextual Features
    daily_sales['sku_popularity_rank'] = daily_sales.groupby(['category_id', 'transaction_date'])['roll_30d'].rank(pct=True)
    cat_momentum = daily_sales.groupby(['category_id', 'transaction_date'])['rolling_sales_growth_7d'].mean().reset_index()
    cat_momentum.rename(columns={'rolling_sales_growth_7d': 'cross_category_momentum'}, inplace=True)
    daily_sales = daily_sales.merge(cat_momentum, on=['category_id', 'transaction_date'])
    
    latest_trends = trends_df.groupby('category_id')['trend_score'].mean().reset_index()
    daily_sales = daily_sales.merge(latest_trends, on='category_id', how='left').fillna({'trend_score': 50})
    
    # 5. Clean up for training
    df = daily_sales.dropna(subset=['rolling_sales_growth_7d', 'rolling_sales_growth_30d', 'sales_acceleration_rate']).copy()
    
    features = ['rolling_sales_growth_7d', 'rolling_sales_growth_30d', 'sku_popularity_rank', 'cross_category_momentum', 'trend_score']
    target = 'sales_acceleration_rate'
    
    X = df[features].values
    y = df[target].values
    
    # 6. Train/Test Split (80/20)
    print(f"Dataset generated: {len(X)} samples.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    # 7. K-Fold Cross Validation
    print("Performing 5-Fold Cross Validation...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_maes = []
    cv_r2s = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]
        
        model_cv = lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31, objective='regression', random_state=42, n_jobs=-1, verbose=-1
        )
        # Using fit directly without early stopping for CV simplify
        model_cv.fit(X_tr, y_tr)
        preds = model_cv.predict(X_val)
        
        cv_maes.append(mean_absolute_error(y_val, preds))
        cv_r2s.append(r2_score(y_val, preds))
        
    print(f"CV MAE: {np.mean(cv_maes):.4f}")
    print(f"CV R2: {np.mean(cv_r2s):.4f}")

    # 8. Final LightGBM Training
    print("Training final model on full train set...")
    final_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=31, max_depth=-1, objective='regression', random_state=42, n_jobs=-1
    )
    final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='mae', callbacks=[lgb.early_stopping(stopping_rounds=50)])
    
    # 9. Evaluation & Outputs
    y_pred = final_model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    report = {
        "model_name": "Trend Intelligence K-Fold LightGBM",
        "training_samples": len(X_train),
        "test_samples": len(X_test),
        "cross_validation_5fold": {
            "mean_MAE": f"{np.mean(cv_maes):.4f}",
            "mean_R2": f"{np.mean(cv_r2s):.4f}"
        },
        "holdout_test_metrics": {
            "MAE": f"{mae:.4f}",
            "R2_Score": f"{r2:.4f}"
        },
        "feature_importances": dict(zip(features, [float(i) for i in final_model.feature_importances_]))
    }
    
    print("\n--- TREND INTELLIGENCE REPORT ---")
    print(json.dumps(report, indent=4))
    
    if not os.path.exists('ml/trend_intelligence'):
        os.makedirs('ml/trend_intelligence')
    final_model.booster_.save_model('ml/trend_intelligence/trend_model.txt')
    with open('ml/trend_intelligence/training_report.json', 'w') as f:
        json.dump(report, f, indent=4)
    print("\n[SUCCESS] Trend Intelligence Model trained and deployed.")

if __name__ == "__main__":
    main()


--- STARTING TREND INTELLIGENCE K-FOLD TRAINING (LightGBM) ---


C:\Users\madha\AppData\Local\Temp\ipykernel_31452\3749537072.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_df = pd.read_sql("SELECT transaction_date, store_id, product_id, units_sold, promotion_flag FROM public.sales_transactions", conn)


C:\Users\madha\AppData\Local\Temp\ipykernel_31452\3749537072.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products_df = pd.read_sql("SELECT product_id, category_id, product_name FROM public.products", conn)
C:\Users\madha\AppData\Local\Temp\ipykernel_31452\3749537072.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  trends_df = pd.read_sql("SELECT category_id, trend_score, momentum FROM public.trends", conn)


Dataset generated: 9210 samples.
Performing 5-Fold Cross Validation...


CV MAE: 0.1025
CV R2: 0.6957
Training final model on full train set...
Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[135]	valid_0's l1: 0.0974031	valid_0's l2: 0.0181269

--- TREND INTELLIGENCE REPORT ---
{
    "model_name": "Trend Intelligence K-Fold LightGBM",
    "training_samples": 7368,
    "test_samples": 1842,
    "cross_validation_5fold": {
        "mean_MAE": "0.1025",
        "mean_R2": "0.6957"
    },
    "holdout_test_metrics": {
        "MAE": "0.0974",
        "R2_Score": "0.7417"
    },
    "feature_importances": {
        "rolling_sales_growth_7d": 1028.0,
        "rolling_sales_growth_30d": 1352.0,
        "sku_popularity_rank": 367.0,
        "cross_category_momentum": 1004.0,
        "trend_score": 299.0
    }
}

[SUCCESS] Trend Intelligence Model trained and deployed.
